In [15]:
# load all the pngs in the folder png

import os
import PIL.Image

pngs = os.listdir('png')
# sort the pngs in numerical order
pngs = sorted(pngs, key=lambda x: int(x.split('.')[0]))
print(pngs)
# load all the pngs with transparency
images = [PIL.Image.open('png/' + png).convert('RGBA') for png in pngs]


['000000.png', '000001.png', '000002.png', '000003.png', '000004.png', '000005.png', '000006.png', '000007.png', '000008.png', '000009.png', '000010.png', '000011.png', '000013.png', '000014.png', '000015.png', '000016.png', '000017.png', '000018.png', '000019.png', '000020.png', '000021.png', '000022.png', '000023.png', '000024.png', '000025.png', '000026.png', '000027.png', '000028.png', '000029.png', '000030.png', '000031.png', '000032.png', '000033.png', '000034.png', '000035.png', '000036.png', '000037.png', '000038.png', '000039.png', '000040.png', '000041.png', '000042.png', '000043.png', '000044.png', '000045.png', '000046.png', '000047.png', '000048.png', '000049.png', '000050.png', '000051.png', '000052.png', '000053.png', '000054.png', '000055.png']


In [24]:

def get_bounding_box(image):
    width, height = image.size
    min_x = width
    min_y = height
    max_x = 0
    max_y = 0
    for x in range(width):
        for y in range(height):
            r, g, b, a = image.getpixel((x, y))
            if a != 0:
                min_x = min(min_x, x)
                min_y = min(min_y, y)
                max_x = max(max_x, x)
                max_y = max(max_y, y)
    return min_x, min_y, max_x, max_y


def center_object(image):
    min_x, min_y, max_x, max_y = get_bounding_box(image)
    width, height = image.size
    new_image = PIL.Image.new('RGBA', (width, height), (0, 0, 0, 0))
    new_image.paste(image, (width // 2 - (max_x + min_x) // 2 ,  height // 2 -  (max_y + min_y) // 2), mask=image)
    return new_image


for i, image in enumerate(images):
    centered = center_object(image)
    centered.save('output/' + "centered_" + pngs[i], "PNG")



In [25]:
# scan png in the output folder and convert to webp
pngs = os.listdir('output')

for png in pngs:
    image = PIL.Image.open('output/' + png)
    image.save('webp/' + png.split('.')[0] + '.webp', "WEBP")

In [31]:
# scan webp in the webp folder and convert to gif

webps = os.listdir('webp')
# sort as string
webps = sorted(webps, key=lambda x: x.split('.')[0])
images = [PIL.Image.open('webp/' + webp) for webp in webps]

images[0].save('output.gif', save_all=True, append_images=images[1:], duration=50, loop=0, disposal=2)

In [28]:
# convert png images to gif, every frame needs to clear the previous frame to avoid ghosting
# each frame is displayed for 100ms
images[0].save('output.gif', save_all=True, append_images=images[1:], duration=100, loop=0)


In [32]:
# load all webp images

webps = os.listdir('webp')
# sort as string
webps = sorted(webps, key=lambda x: x.split('.')[0])
images = [PIL.Image.open('webp/' + webp) for webp in webps]


def get_bounding_box(image):
    width, height = image.size
    min_x = width
    min_y = height
    max_x = 0
    max_y = 0
    for x in range(width):
        for y in range(height):
            r, g, b, a = image.getpixel((x, y))
            if a != 0:
                min_x = min(min_x, x)
                min_y = min(min_y, y)
                max_x = max(max_x, x)
                max_y = max(max_y, y)
    return min_x, min_y, max_x, max_y

min_x_of_all = 100000000
min_y_of_all = 100000000
max_x_of_all = 0
max_y_of_all = 0

for i, image in enumerate(images):
    min_x, min_y, max_x, max_y = get_bounding_box(image)
    min_x_of_all = min(min_x_of_all, min_x)
    min_y_of_all = min(min_y_of_all, min_y)
    max_x_of_all = max(max_x_of_all, max_x)
    max_y_of_all = max(max_y_of_all, max_y)

print(min_x_of_all, min_y_of_all, max_x_of_all, max_y_of_all)

403 293 2980 3796


In [34]:
# crop all the images to the same size
for i, image in enumerate(images):
    width, height = image.size
    new_image = PIL.Image.new('RGBA', (max_x_of_all - min_x_of_all + 1, max_y_of_all - min_y_of_all + 1), (0, 0, 0, 0))
    new_image.paste(image, (-min_x_of_all, -min_y_of_all), mask=image)
    new_image.save('cropped/' + "cropped_" + webps[i], "WEBP")

In [35]:
# read all the cropped images

cropped_webps = os.listdir('cropped')
# sort as string
cropped_webps = sorted(cropped_webps, key=lambda x: x.split('.')[0])
images = [PIL.Image.open('cropped/' + cropped_webp) for cropped_webp in cropped_webps]

for i, image in enumerate(images):
  # resize width to 1024
  width, height = image.size
  new_height = 1024 * height // width
  image = image.resize((1024, new_height))
  # compress the image
  image.save('resized/' + "resized_" + cropped_webps[i], "WEBP", optimize=True, quality=50)

In [47]:
# read all the cropped images

cropped_webps = os.listdir('cropped')
# sort as string
cropped_webps = sorted(cropped_webps, key=lambda x: x.split('.')[0])
images = [PIL.Image.open('cropped/' + cropped_webp) for cropped_webp in cropped_webps]


resized_images = []

for i, image in enumerate(images):
  # resize height to 32
  width, height = image.size
  new_width = 32 * width // height
  image = image.resize((new_width, 32))

  # make the image a square
  new_image = PIL.Image.new('RGBA', (32, 32), (0, 0, 0, 0))
  new_image.paste(image, ( (32 - new_width) // 2, 0), mask=image)
  resized_images.append(new_image)

  # save in temp

  new_image.save('temp/' + "temp_" + cropped_webps[i] +".png", "PNG", optimize=True, quality=50)

# read all the temp images
temp_images = os.listdir('temp')
# sort as string
temp_images = sorted(temp_images, key=lambda x: x.split('.')[0])

temp_images = ["temp/" + temp_image for temp_image in temp_images]




from apng import APNG

# skip every other frame

temp_images = [temp_images[i] for i in range(0, len(temp_images), 2)]



APNG.from_files(temp_images, delay=100).save('output.png')

# convert png images to apng, every frame needs to clear the previous frame to avoid ghosting
# each frame is displayed for 100ms
# resized_images[0].save('output.gif', save_all=True, append_images=resized_images[1:], duration=50, loop=0, disposal=2)
